In [3]:
import logging
import time
from datetime import datetime
import warnings
import shutil
import json
import pickle
import torch
import sys

import os
import pandas as pd
import numpy as np
import networkx as nx
import scanpy as sc
import anndata as ad

import torch.nn as nn
from tqdm import tqdm
from collections import Counter
from itertools import islice

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2

import sys
sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba")
sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src")

from src.models.DyGMamba import DyGMamba
from src.models.modules import MergeLayer, MergeLayerTD

from src.utils.load_configs import load_link_prediction_args

from src.utils.DataLoader import get_model_data
from src.utils.DataLoader import get_idx_data_loader
from src.utils.utils import get_neighbor_sampler, NegativeEdgeSampler
from src.utils.utils import get_parameter_sizes
from src.utils.utils import set_random_seed
from src.utils.utils import convert_to_gpu, create_optimizer
from src.utils.EarlyStopping import EarlyStopping
from src.utils.metrics import get_link_prediction_metrics
from src.models.evaluate_models_utils import evaluate_model_link_prediction
from src.models.inference_grn import model_link_prediction
from src.data_preprocess import filter_jaspar_tf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Configuration

In [15]:
import os
print("********************** start ********************")

start_time = time.time()  # start the time
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")

# get arguments
args = load_link_prediction_args(is_evaluation=False)

print("**********************device********************")
print(f"Now use device is {args.device}")

org_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"

dyg_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result2/"

assess_result = "/home/liyang/BioWuYan/dygmamba_project/data/assess_result/"
# os.makedirs(data_path, exist_ok = True)

********************** start ********************
[2025-12-24 13:44:37] Start the job
**********************device********************
Now use device is cuda:0


# Load Data

In [5]:
adata_atac = ad.read_h5ad(dyg_result_path + "atac.h5ad")

adata_rna = ad.read_h5ad(dyg_result_path + "rna.h5ad")

feat_path = dyg_result_path + "edge_features.npy"
edge_label_path = dyg_result_path + "edge_labels.npy"

Edge_feature = np.load(feat_path, mmap_mode="r")
Edge_feature = Edge_feature.reshape(-1,1).copy()
Edge_label = np.load(edge_label_path, mmap_mode="r")
Edge_label = Edge_label.reshape(-1,1).copy()

with open(dyg_result_path + "node_feature_data.pkl", "rb") as f:
    load_data = pickle.load(f)

Node_feature = load_data['node_feature']

Node_id = pd.read_pickle(dyg_result_path + "node_id.pkl")

graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

# Result Analysis

## Benchmark

In [60]:
from src.data_preprocess import adata_to_dataframe

benchmark_data_path = org_data_path

benchmark_tf_gene_grn = ad.read_h5ad(benchmark_data_path + "tf_gene_network.h5ad")

benchmark_peak_gene_grn = ad.read_h5ad(benchmark_data_path + "peak_gene_network.h5ad")

benchmark_tf_peak_grn = ad.read_h5ad(benchmark_data_path + "tf_peak_network.h5ad")

benchmark_tf_peak_df = adata_to_dataframe(benchmark_tf_peak_grn)
benchmark_tf_peak_df = benchmark_tf_peak_df.rename(columns= {"obs":"Peak", "var":"TF"})
print(benchmark_tf_peak_df)

benchmark_peak_gene_df = adata_to_dataframe(benchmark_peak_gene_grn)
benchmark_peak_gene_df = benchmark_peak_gene_df.rename(columns= {"obs":"Peak", "var":"Gene"})
print(benchmark_peak_gene_df)

benchmark_tf_gene_df = adata_to_dataframe(benchmark_tf_gene_grn)

benchmark_tf_gene_df = benchmark_tf_gene_df.rename(columns= {"obs":"TF", "var":"Gene"})
print(benchmark_tf_gene_df)
benchmark_tf_gene_threshold = 5

                             Peak     TF  value
0        chr1-100028461-100029069  CREB1      1
1        chr1-100028461-100029069   CREM      1
2        chr1-100028461-100029069   E2F1      1
3        chr1-100028461-100029069   ETV6      1
4        chr1-100028461-100029069  FOSL1      1
...                           ...    ...    ...
2758834      chrX-9996978-9998203  TBX21      1
2758835      chrX-9996978-9998203  TCF12      1
2758836      chrX-9996978-9998203   TCF3      1
2758837      chrX-9996978-9998203    YY1      1
2758838    chrY-11310224-11310574  CREB1      1

[2758839 rows x 3 columns]
                             Peak     Gene  value
0        chr1-100028461-100029069      AGL      1
1        chr1-100028461-100029069     ARNT      1
2        chr1-100028461-100029069     BCL9      1
3        chr1-100028461-100029069     COPA      1
4        chr1-100028461-100029069  CTDSPL2      1
...                           ...      ...    ...
1686605      chrX-9996978-9998203     TLR7    

In [61]:
print(benchmark_peak_gene_grn)
print(f"Peak-Gene: {benchmark_peak_gene_df['Peak'].nunique()}, \
    {benchmark_peak_gene_df['Gene'].nunique()}, edge: {len(benchmark_peak_gene_df)}")

print(benchmark_tf_gene_grn)
print(f"TF-Gene: {benchmark_tf_gene_df['TF'].nunique()}, \
    {benchmark_tf_gene_df['Gene'].nunique()}, edge: {len(benchmark_tf_gene_df)}")


AnnData object with n_obs × n_vars = 71541 × 2000
    uns: 'description'
Peak-Gene: 71541,     2000, edge: 1686610
AnnData object with n_obs × n_vars = 112 × 2000
    uns: 'description'
TF-Gene: 112,     2000, edge: 223908


## TF-region data

In [26]:

jaspar_tf_region_file = org_data_path + "jaspar_data.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)


coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})


print("*"*50)
print(adata_region_tf)
print("*"*50)
print("*"*50)
print(tf_peak_df)
print("*"*50)
print("*"*50)
print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print("*"*50)

/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  > 最终形状: (72563, 879)
**************************************************
AnnData object with n_obs × n_vars = 72563 × 879
    uns: 'description'
**************************************************
**************************************************
                            Peak              TF  value
0               chr1-10231-10656           Dmrt1      1
1               chr1-10231-10656          ZNF708      1
2               chr1-10231-10656            CDX2      1
3               chr1-10231-10656            Lef1      1
4               chr1-10231-10656             SP1      1
...                          ...             ...    ...
55056385  chrY-11310224-11310574  Stat5a::Stat5b      1
55056386  chrY-11310224-11310574           GATA2      1
55056387  chrY-11310224-11310574     ETV2::FOXI1      1
55056388  chrY-11310224-11310574         Bhlha15      1
55056389  chrY-11310224-11310574           PRDM9      1

[55056390 rows x 3 columns]
**************************************************


In [32]:
dygmamba_tf_peak_df = tf_peak_df
dygmamba_tf_peak_df = dygmamba_tf_peak_df.rename(columns={"value":"predict"})

total_tf = set(benchmark_tf_peak_df["TF"]) & set(dygmamba_tf_peak_df["TF"])
benchmark_tf_peak_df = benchmark_tf_peak_df[benchmark_tf_peak_df["TF"].isin(total_tf)].copy()
dygmamba_tf_peak_df = dygmamba_tf_peak_df[dygmamba_tf_peak_df["TF"].isin(total_tf)].copy()

total_peak = set(adata_atac.var_names)
benchmark_tf_peak_df = benchmark_tf_peak_df[benchmark_tf_peak_df["Peak"].isin(total_peak)].copy()
dygmamba_tf_peak_df = dygmamba_tf_peak_df[dygmamba_tf_peak_df["Peak"].isin(total_peak)].copy()


dyg_merged_tf_peak_data = pd.merge(benchmark_tf_peak_df, dygmamba_tf_peak_df, on = ["TF", "Peak"], how="outer").fillna(0)

print(dyg_merged_tf_peak_data)
print(dygmamba_tf_peak_df)

print("*"*50)
print(f"Merged TF-Peak: {dyg_merged_tf_peak_data['TF'].nunique()}, {dyg_merged_tf_peak_data['Peak'].nunique()}, \
    {len(dyg_merged_tf_peak_data)}")

print(f"Dygmamba TF-Peak: {dygmamba_tf_peak_df['TF'].nunique()}, {dygmamba_tf_peak_df['Peak'].nunique()},\
    edge: {len(dygmamba_tf_peak_df)}")

print(f"Benchmark TF-Peak: {benchmark_tf_peak_df['TF'].nunique()}, \
    {benchmark_tf_peak_df['Peak'].nunique()}, edge: {len(benchmark_tf_peak_df)}")
print("*"*50)

                            Peak      TF  value  predict
0       chr1-100037313-100039097    ATF2    1.0      1.0
1         chr1-10032558-10033598    ATF2    0.0      1.0
2       chr1-100351277-100353494    ATF2    1.0      1.0
3       chr1-100894820-100896914    ATF2    0.0      1.0
4       chr1-101235525-101239028    ATF2    1.0      1.0
...                          ...     ...    ...      ...
533745    chrX-53683390-53684529  ZNF740    1.0      1.0
533746    chrX-53714219-53717438  ZNF740    0.0      1.0
533747    chrX-54043829-54044967  ZNF740    0.0      1.0
533748    chrX-64204939-64206473  ZNF740    1.0      1.0
533749      chrX-7147230-7148785  ZNF740    0.0      1.0

[533750 rows x 4 columns]
                              Peak      TF  predict
516             chr1-629315-630015   ESRRA        1
519             chr1-629315-630015     JUN        1
533             chr1-629315-630015   GABPA        1
535             chr1-629315-630015   NR2F1        1
539             chr1-629315-6

In [33]:

benchmark_result = []
result_type = "binary"
beta_value = 1

dyg_y_true = dyg_merged_tf_peak_data["value"].astype(int)
dyg_y_pre = dyg_merged_tf_peak_data["predict"].astype(int)
dyg_model_name = "Dygmamba_peak"

dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)
dyg_tf_region_result = pd.DataFrame([dyg_dict])
print(dyg_tf_region_result)



      model_name  TN     FP     FN      TP  precision    recall  FPR  \
0  Dygmamba_peak   0  96886  39486  397378   0.803979  0.909615  1.0   

        AUC   f_score  
0  0.454807  0.853541  


## Region-gene

In [37]:

adata_rp_gene_peak = ad.read_h5ad(dyg_result_path + "rp_gene_peak.h5ad")
print(adata_rp_gene_peak)
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)

AnnData object with n_obs × n_vars = 500 × 5000
    uns: 'decay_distance', 'description', 'max_range'
        Gene                       Peak  value
0     NDUFS5     chr1-38990697-38992620      1
1       DPP9      chr19-4790997-4792145      1
2    TXNDC15   chr5-134904464-134905833      1
3      PPRC1  chr10-102055526-102056382      1
4      PPRC1  chr10-102064962-102066074      1
..       ...                        ...    ...
437     ELF2   chr4-139176167-139178479      1
438      IVD    chr15-40440044-40441714      1
439    CHTOP   chr1-153670613-153672079      1
440    KIF3A   chr5-132662988-132664501      1
441    KIF3A   chr5-132674425-132675608      1

[442 rows x 3 columns]


In [34]:
graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_graph = New_Graph.copy()

result_path = dyg_result_path + 'my_result_run{run}.npy'

predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)

result_graph["predict"] = predict_edge_label
predict_grn = result_graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)

peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()

print("*"*50)
print(peak_gene_df)
print("*"*50)

print("*"*50)
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print("*"*50)


**************************************************
                          Peak    Gene         ts   predict
0      chr12-53370831-53371774    AAAS   0.103799  0.054033
1      chr12-53370831-53371774    AAAS   0.130393  1.000000
2      chr12-53370831-53371774    AAAS   0.427715  1.000000
3      chr12-53370831-53371774    AAAS   0.487691  1.000000
4      chr12-53370831-53371774    AAAS   0.560851  1.000000
...                        ...     ...        ...       ...
37223  chr20-45933784-45935583  ZSWIM1   9.705171  1.000000
37224  chr20-45933784-45935583  ZSWIM1   9.757383  1.000000
37225  chr20-45933784-45935583  ZSWIM1  10.113001  1.000000
37226  chr20-45933784-45935583  ZSWIM1  10.506163  1.000000
37227  chr20-45933784-45935583  ZSWIM1  10.729193  1.000000

[37228 rows x 4 columns]
**************************************************
**************************************************
Peak-Gene: 412, 246, edge:37228
**************************************************


In [38]:

dygmamba_peak_gene_grn = peak_gene_df

avg_active_peak_gene_grn = dygmamba_peak_gene_grn.groupby(['Peak', 'Gene']).agg(
    avg_ts_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()



avg_active_peak_gene_grn = avg_active_peak_gene_grn[avg_active_peak_gene_grn["Peak"].isin(total_peak)].copy()
benchmark_peak_gene_df = benchmark_peak_gene_df[benchmark_peak_gene_df["Peak"].isin(total_peak)].copy()

dyg_merged_peak_gene_data = pd.merge(benchmark_peak_gene_df, avg_active_peak_gene_grn, on = ["Gene", "Peak"], how="outer").fillna(0)

print("*"*50)
print(avg_active_peak_gene_grn)
print("*"*50)
print(dyg_merged_peak_gene_data)

print("*"*50)
print(f"Merged Peak-Gene: {dyg_merged_peak_gene_data['Peak'].nunique()}, {dyg_merged_peak_gene_data['Gene'].nunique()}, \
    {len(dyg_merged_peak_gene_data)}")

print(f"Dygmamba Peak-Gene: {avg_active_peak_gene_grn['Peak'].nunique()}, {avg_active_peak_gene_grn['Gene'].nunique()}, \
    edge: {len(avg_active_peak_gene_grn)}")

print(f"Benchmark Peak-Gene: {benchmark_peak_gene_df['Peak'].nunique()}, \
    {benchmark_peak_gene_df['Gene'].nunique()}, edge: {len(benchmark_peak_gene_df)}")
print("*"*50)



**************************************************
                         Peak     Gene  avg_ts_weight  avg_total_weight
0      chr1-10209868-10211197    KIF1B       0.988301         79.064110
1      chr1-10985220-10986808      SRM       0.983288         55.064110
2      chr1-11011896-11013220      SRM       0.983107         56.037113
3      chr1-11059602-11060685      SRM       0.987037         73.040741
4    chr1-111139381-111140734  DENND2D       0.983581         56.064110
..                        ...      ...            ...               ...
437    chr9-99253171-99254353   SEC61B       0.986436         68.064110
438  chrX-153926220-153928652    HCFC1       0.987686         75.064110
439    chrX-47144634-47145544    CDK16       0.984402         59.064110
440    chrX-48938639-48940136    PQBP1       0.988153         78.064110
441    chrX-48957411-48958710    PQBP1       0.985012         63.040741

[442 rows x 4 columns]
**************************************************
          

In [39]:

dyg_merged_peak_gene_data["predict"] = (dyg_merged_peak_gene_data["avg_ts_weight"]>0.9).astype(int)

dyg_y_true = dyg_merged_peak_gene_data["value"].astype(int)
dyg_y_pre = dyg_merged_peak_gene_data["predict"].astype(int)
dyg_model_name = "Dygmamba_peak_gene"

dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

dyg_peak_gene_result = pd.DataFrame([dyg_dict])
print(dyg_peak_gene_result)

print(f"merged: {len(dyg_merged_peak_gene_data)}, benchmark peak gene: {len(benchmark_peak_gene_df)},\
    dyg peak gene {len(avg_active_peak_gene_grn)}")

print("*"*50)
print(f"Peak-Gene: {dyg_merged_peak_gene_data['Peak'].nunique()}, \
    {dyg_merged_peak_gene_data['Gene'].nunique()}, edge:{len(dyg_merged_peak_gene_data)}")
print("*"*50)

           model_name  TN  FP      FN   TP  precision    recall  FPR  AUC  \
0  Dygmamba_peak_gene   0   0  191343  442        1.0  0.002305  NaN  NaN   

    f_score  
0  0.004599  
merged: 191785, benchmark peak gene: 191785,    dyg peak gene 442
**************************************************
Peak-Gene: 5000,     1999, edge:191785
**************************************************


/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/analysis/assess.py:56: RuntimeWarning: invalid value encountered in scalar divide
  model_FPR = float(model_FP/(model_FP+model_TN))


## TF-gene network

In [47]:
dygmamba_tf_peak_df = dygmamba_tf_peak_df.rename(columns={"predict":"value"})

peak_gene_grn = peak_gene_df[peak_gene_df["Peak"].isin(total_peak)]
print(peak_gene_grn)

merged_df = pd.merge(dygmamba_tf_peak_df, peak_gene_grn, on='Peak')
print(merged_df)

                          Peak    Gene         ts   predict
0      chr12-53370831-53371774    AAAS   0.103799  0.054033
1      chr12-53370831-53371774    AAAS   0.130393  1.000000
2      chr12-53370831-53371774    AAAS   0.427715  1.000000
3      chr12-53370831-53371774    AAAS   0.487691  1.000000
4      chr12-53370831-53371774    AAAS   0.560851  1.000000
...                        ...     ...        ...       ...
37223  chr20-45933784-45935583  ZSWIM1   9.705171  1.000000
37224  chr20-45933784-45935583  ZSWIM1   9.757383  1.000000
37225  chr20-45933784-45935583  ZSWIM1  10.113001  1.000000
37226  chr20-45933784-45935583  ZSWIM1  10.506163  1.000000
37227  chr20-45933784-45935583  ZSWIM1  10.729193  1.000000

[37228 rows x 4 columns]
                             Peak     TF  value      Gene         ts   predict
0              chr1-629315-630015  ESRRA      1  MTCO1P12   0.000000  0.064111
1              chr1-629315-630015  ESRRA      1  MTCO1P12   0.103799  0.999998
2              ch

In [54]:


tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print("*"*50)
print(tf_gene_grn)
print("*"*50)

print(f"Dygmamba TF-Peak: {dygmamba_tf_peak_df['TF'].nunique()}, {dygmamba_tf_peak_df['Peak'].nunique()},\
    edge: {len(dygmamba_tf_peak_df)}")

print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")

print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")

tf_gene_grn.to_pickle(dyg_result_path + "new_tf_gene_grn_1224.pkl")


**************************************************
             TF    Gene         ts  peak_num  avg_weight  total_weight
0          ATF2    AAAS   0.000000         1    0.064111      0.064111
1          ATF2    AAAS   0.103799         2    0.527016      1.054032
2          ATF2    AAAS   0.130393         2    1.000000      2.000000
3          ATF2    AAAS   0.427715         2    1.000000      2.000000
4          ATF2    AAAS   0.487691         2    1.000000      2.000000
...         ...     ...        ...       ...         ...           ...
2510266  ZNF740  ZSWIM1  10.038790         1    1.000000      1.000000
2510267  ZNF740  ZSWIM1  10.113001         1    1.000000      1.000000
2510268  ZNF740  ZSWIM1  10.506163         1    1.000000      1.000000
2510269  ZNF740  ZSWIM1  10.584224         1    1.000000      1.000000
2510270  ZNF740  ZSWIM1  10.729193         1    1.000000      1.000000

[2510271 rows x 6 columns]
**************************************************
Dygmamba TF-Peak: 

### Average GRN


In [55]:
# tf_gene_grn = pd.read_pickle(dyg_result_path + "new_tf_gene_grn_1223.pkl")

avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")

print(avg_active_tf_gene_grn)


pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

print(avg_global_tf_gene_grn)

print("*"*50)
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")
print(f"avg TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()},\
    edge: {len(avg_active_tf_gene_grn)}")

           TF      Gene  avg_ts_weight  avg_total_weight
0        ATF2      AAAS       1.333969        248.118149
1        ATF2  AASDHPPT       0.987846         76.064110
2        ATF2     ABHD6       0.983581         56.064110
3        ATF2      ACO2       1.557796        288.192322
4        ATF2     ADCY3       0.989601         89.064110
...       ...       ...            ...               ...
21953  ZNF740    ZNF473       0.988724         82.064110
21954  ZNF740    ZNF581       1.757182        346.164856
21955  ZNF740    ZNF598       1.170140        152.118149
21956  ZNF740    ZNF672       1.455715        256.205841
21957  ZNF740    ZSWIM1       1.263321        178.128220

[21958 rows x 4 columns]
           TF      Gene  average_active_weight
0        ATF2      AAAS               1.112637
1        ATF2  AASDHPPT               0.341095
2        ATF2     ABHD6               0.251409
3        ATF2      ACO2               1.292342
4        ATF2     ADCY3               0.399391
...     

### Merge

In [50]:
benchmark_tf_gene_df = benchmark_tf_gene_df[benchmark_tf_gene_df["TF"].isin(total_tf)].copy()

dyg_avg_active_tf_gene_grn = avg_active_tf_gene_grn
dyg_avg_global_tf_gene_grn = avg_global_tf_gene_grn
dyg_avg_active_tf_gene_grn.columns.name = ""
dyg_avg_active_tf_gene_grn = dyg_avg_active_tf_gene_grn.reset_index()
dyg_avg_active_tf_gene_grn = dyg_avg_active_tf_gene_grn.drop(["index"],axis = 1)
dyg_merged_data = pd.merge(benchmark_tf_gene_df, dyg_avg_active_tf_gene_grn, on = ["TF", "Gene"], how="outer").fillna(0)

dyg_avg_global_tf_gene_grn.columns.name = ""
dyg_avg_global_tf_gene_grn = dyg_avg_global_tf_gene_grn.reset_index()
dyg_avg_global_tf_gene_grn = dyg_avg_global_tf_gene_grn.drop(["index"],axis = 1)
dyg_global_merged_data = pd.merge(benchmark_tf_gene_df, dyg_avg_global_tf_gene_grn, on = ["TF", "Gene"], how="outer").fillna(0)

In [57]:
from src.analysis.assess import dygmamba_assess
benchmark_tf_gene_threshold = 200
threshold_weight_global = 0.2
threshold_weight_active = 0.2
benchmark_result = []
result_type = "binary"
beta_value = 1

dyg_merged_data["label"] = (dyg_merged_data["value"] > benchmark_tf_gene_threshold).astype(int)
dyg_merged_data["predict_label"] = (dyg_merged_data["avg_ts_weight"]> threshold_weight_active).astype(int)
dyg_merge_grn = dyg_merged_data.copy()
dyg_y_true = dyg_merge_grn["label"].astype(int)
dyg_y_pre = dyg_merge_grn["predict_label"].astype(int)
dyg_model_name = "Dygmamba"
dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

benchmark_result.append(dyg_dict)
# dyg_gold_grn, dyg_gold_grn = analysis_tf_recovery(dyg_merged_data, 
#                                                     model_name = dyg_model_name, fig_path = assess_result)

dyg_global_merged_data["label"] = (dyg_global_merged_data["value"] > benchmark_tf_gene_threshold).astype(int)
dyg_global_merged_data["predict_label"] = (dyg_global_merged_data["average_active_weight"]> threshold_weight_global).astype(int)

dyg_global_merge_grn = dyg_global_merged_data.copy()

dyg_global_y_true = dyg_global_merge_grn["label"].astype(int)
dyg_global_y_pre = dyg_global_merge_grn["predict_label"].astype(int)
dyg_global_model_name = "Dygmamba" + "_global"

dyg_global_dict = dygmamba_assess(dyg_global_y_true, dyg_global_y_pre, model_name = dyg_global_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

benchmark_result.append(dyg_global_dict)
    
# dyg_global_gold_grn, dyg_global_gold_grn = analysis_tf_recovery(dyg_global_merged_data, 
#                                                     model_name = dyg_global_model_name, fig_path = assess_result)

benchmark_result_df = pd.DataFrame(benchmark_result)

print(benchmark_result_df)


active_num = dyg_merged_data["label"].sum(axis=0)
active_total = len(dyg_merged_data)

global_num = dyg_global_merged_data["label"].sum(axis=0)
global_total = len(dyg_global_merged_data)

print(f"active_num: {active_num}/{active_total}; global num: {global_num}/{global_total}")

print(f"benchmark tf gene: {len(benchmark_tf_gene_df)}, dyg global tf gene: {len(dyg_avg_global_tf_gene_grn)},\
    merge: {len(dyg_global_merged_data)}")

print("*"*50)

print(f"Benchmark TF-Gene: {benchmark_tf_gene_df['TF'].nunique()}, \
    {benchmark_tf_gene_df['Gene'].nunique()}, edge: {len(benchmark_tf_gene_df)}")

print(f"Dygmamba Merge TF-Gene: {dyg_merge_grn['TF'].nunique()}, \
    {dyg_merge_grn['Gene'].nunique()}, edge: {len(dyg_merge_grn)}")

print(f"Dygmamba Global Merge TF-Gene: {dyg_global_merged_data['TF'].nunique()}, \
    {dyg_global_merged_data['Gene'].nunique()}, edge: {len(dyg_global_merged_data)}")

print(f"Dygmamba Global TF-Gene: {dyg_avg_global_tf_gene_grn['TF'].nunique()}, \
    {dyg_avg_global_tf_gene_grn['Gene'].nunique()}, edge: {len(dyg_avg_global_tf_gene_grn)}")

print(f"Dygmamba active TF-Gene: {dyg_avg_active_tf_gene_grn['TF'].nunique()}, \
    {dyg_avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(dyg_avg_active_tf_gene_grn)}")
print("*"*50)





        model_name     TN    FP      FN     TP  precision    recall       FPR  \
0         Dygmamba  49368  3632  124593  18326   0.834593  0.128226  0.068528   
1  Dygmamba_global  49368  3632  124593  18326   0.834593  0.128226  0.068528   

        AUC   f_score  
0  0.529849  0.222299  
1  0.529849  0.222299  
active_num: 142919/195919; global num: 142919/195919
benchmark tf gene: 195919, dyg global tf gene: 21958,    merge: 195919
**************************************************
Benchmark TF-Gene: 98,     2000, edge: 195919
Dygmamba Merge TF-Gene: 98,     2000, edge: 195919
Dygmamba Global Merge TF-Gene: 98,     2000, edge: 195919
Dygmamba Global TF-Gene: 98,     246, edge: 21958
Dygmamba active TF-Gene: 98,     246, edge: 21958
**************************************************


# Total Code

In [36]:
graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']


result_path = dyg_result_path + 'my_result_run{run}.npy'
predict_edge_label = np.load(result_path)
New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)
peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()
print("*"*20)
print(peak_gene_df)


jaspar_tf_region_file = org_data_path + "jaspar_data.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)

coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})


merged_df = pd.merge(tf_peak_df, peak_gene_df, on='Peak')
print("*"*50)
print(merged_df)

tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print("*"*50)
print(tf_gene_grn)

tf_gene_grn.to_pickle(dyg_result_path + "new_tf_gene_grn.pkl")


avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")

print("*"*50)
print(avg_active_tf_gene_grn)


pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

print("*"*50)
print(avg_global_tf_gene_grn)

print("*"*50)
print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")
print(f"TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(avg_active_tf_gene_grn)}")

********************
                          Peak    Gene         ts   predict
0      chr12-53370831-53371774    AAAS   0.103799  0.054033
1      chr12-53370831-53371774    AAAS   0.130393  1.000000
2      chr12-53370831-53371774    AAAS   0.427715  1.000000
3      chr12-53370831-53371774    AAAS   0.487691  1.000000
4      chr12-53370831-53371774    AAAS   0.560851  1.000000
...                        ...     ...        ...       ...
37223  chr20-45933784-45935583  ZSWIM1   9.705171  1.000000
37224  chr20-45933784-45935583  ZSWIM1   9.757383  1.000000
37225  chr20-45933784-45935583  ZSWIM1  10.113001  1.000000
37226  chr20-45933784-45935583  ZSWIM1  10.506163  1.000000
37227  chr20-45933784-45935583  ZSWIM1  10.729193  1.000000

[37228 rows x 4 columns]


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  > 最终形状: (72563, 879)
********************
                              Peak         TF  value      Gene         ts  \
0               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.000000   
1               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.103799   
2               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.130393   
3               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.162789   
4               chr1-629315-630015  FOSB::JUN      1  MTCO1P12   0.306248   
...                            ...        ...    ...       ...        ...   
29518733  chrX-153926220-153928652      PRDM9      1     HCFC1   9.813189   
29518734  chrX-153926220-153928652      PRDM9      1     HCFC1  10.318362   
29518735  chrX-153926220-153928652      PRDM9      1     HCFC1  10.506163   
29518736  chrX-153926220-153928652      PRDM9      1     HCFC1  10.584224   
29518737  chrX-153926220-153928652      PRDM9      1     HCFC1  10.638032   

           predict  
0         

# Further

In [45]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"
adata_rp_gene_peak = ad.read_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")

In [46]:
print(adata_rp_gene_peak)
from data_preprocess import adata_to_dataframe

prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df.head()
prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)

AnnData object with n_obs × n_vars = 2000 × 71541
    uns: 'decay_distance', 'description', 'max_range'
       Gene                      Peak  value
0      AAAS   chr12-53251618-53252739      1
1      AAAS   chr12-53267768-53268827      1
2      AAAS   chr12-53295185-53295894      1
3      AAAS   chr12-53299560-53300133      1
4      AAAS   chr12-53336297-53336656      1
...     ...                       ...    ...
12237  ZXDC  chr3-126522381-126522675      1
12238   ZYX  chr7-143327917-143328230      1
12239   ZYX  chr7-143362369-143362688      1
12240   ZYX  chr7-143380273-143381719      1
12241   ZYX  chr7-143408952-143409231      1

[12242 rows x 3 columns]


In [49]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result2/"

adata_rp_gene_peak = ad.read_h5ad(output_path + "rp_gene_peak.h5ad")
print(adata_rp_gene_peak)
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)


AnnData object with n_obs × n_vars = 500 × 5000
    uns: 'decay_distance', 'description', 'max_range'
        Gene                       Peak  value
0     NDUFS5     chr1-38990697-38992620      1
1       DPP9      chr19-4790997-4792145      1
2    TXNDC15   chr5-134904464-134905833      1
3      PPRC1  chr10-102055526-102056382      1
4      PPRC1  chr10-102064962-102066074      1
..       ...                        ...    ...
437     ELF2   chr4-139176167-139178479      1
438      IVD    chr15-40440044-40441714      1
439    CHTOP   chr1-153670613-153672079      1
440    KIF3A   chr5-132662988-132664501      1
441    KIF3A   chr5-132674425-132675608      1

[442 rows x 3 columns]
